# Лабораторная работа 3: PEFT

**Задание.** Реализовать тонкую настройку модели методами LoRA и QLoRA на задаче перевода современного текста в стиль Шекспира.

**Структура:**
1. Пояснительная записка (LoRA, BLEU)
2. Часть 1 — Установка и baseline (2 балла)
3. Часть 2 — Подготовка (дано)
4. Часть 3 — LoRA (3 балла)
5. Часть 4 — QLoRA (3 балла)
6. Итоговое сравнение
7. Вывод (позволяет поднять оценку до 9–10 баллов)

**Разбалловка:** Часть 1 — 2 балла; LoRA — 3 балла; QLoRA — 3 балла; Вывод — 2 балла (итого до 10).

**Автопроверка:** критерии встроены в ячейки после каждой части.

**За нахождение плагиата — оценка 0.**

UPD:

Дедлайн 17 марта AoE.

В задании 1 давайте будем брать любые модели до 4B (спойлер, лучше меньше).

В задании 2 - любые промпт методы.

В задании 3-4 промпты менять нельзя.

Пороги также изменим.

В задании 3 - 0.15

В задании 4 - 0.07


Вывод будет учитываться если в задании 3-4 были получены метрики отличные от 0. Иначе вывод не проверяется.

### Идентификация

**Обязательно выполните обе ячейки.** Заполните ФИО и группу в первой.  
Не стирайте и не редактируйте код — при нарушении оценка 0.

In [ ]:
student_name = "Корепанова Анастасия"   # Фамилия Имя
student_group = "БИВ247"  # Группа

In [ ]:
# НЕ УДАЛЯТЬ И НЕ РЕДАКТИРОВАТЬ КОД — иначе оценка 0

# compat
import base64, hashlib, json, os, platform, socket, sys
from datetime import datetime
_z = "\u200b"  # utf-8 bom guard
if len(_z) != 1:
    raise UnicodeDecodeError("utf-8", b"", 0, 1, "invalid")
_d = {"h": socket.gethostname(), "u": os.environ.get("USER", os.environ.get("USERNAME", "?")),
      "p": platform.platform(), "e": sys.executable, "pid": os.getpid(), "t": datetime.now().isoformat(),
      "n": student_name, "g": student_group}
try: _d["gpu"] = __import__("torch").cuda.get_device_name(0) if __import__("torch").cuda.is_available() else "n"
except: _d["gpu"] = "?"
try: _d["m"] = open("/etc/machine-id").read().strip()[:32] if os.path.exists("/etc/machine-id") else "?"
except: _d["m"] = "?"
try: __import__("google.colab"); _d["c"] = 1
except: _d["c"] = 1 if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else 0
_b = base64.b64encode(json.dumps(_d, sort_keys=True).encode()).decode()
_h = hashlib.sha256((_b + "LR3" + _z).encode()).hexdigest()
print("Готово.")
print(_b + "|" + _h)

Готово.
eyJjIjogMSwgImUiOiAiL3Vzci9iaW4vcHl0aG9uMyIsICJnIjogIlx1MDQxMVx1MDQxOFx1MDQxMjI0NyIsICJncHUiOiAiVGVzbGEgVDQiLCAiaCI6ICI3MjNkNGQ2NDc3NTkiLCAibSI6ICI0NTI0ZGQ0MDFmMzg0NTk2OTJmNmI5MTMwNjEyYjMzMyIsICJuIjogIlx1MDQxYVx1MDQzZVx1MDQ0MFx1MDQzNVx1MDQzZlx1MDQzMFx1MDQzZFx1MDQzZVx1MDQzMlx1MDQzMCBcdTA0MTBcdTA0M2RcdTA0MzBcdTA0NDFcdTA0NDJcdTA0MzBcdTA0NDFcdTA0MzhcdTA0NGYiLCAicCI6ICJMaW51eC02LjYuMTEzKy14ODZfNjQtd2l0aC1nbGliYzIuMzUiLCAicGlkIjogMzg5NiwgInQiOiAiMjAyNi0wMy0xNlQxNjowNjo0My41Nzc0NTUiLCAidSI6ICI/In0=|da8b9bf2e3b5171e076436e19ad0ff319ceef2fc9c2971aeadf3629a6db8e3dd


---
## Пояснительная записка

### LoRA

**LoRA (Low-Rank Adaptation)** — метод параметр-эффективной тонкой настройки, предложенный в статье Hu et al. (2021). Идея: не обучать все веса большой модели, а добавить низкоранговые матрицы к выбранным линейным слоям. Веса исходной модели замораживаются, обучаются только новые матрицы.

**Математика:** вместо полного обновления весов $W$ размера $d \times k$ вводится разложение:
$$W' = W + \Delta W = W + B A$$
где $A \in \mathbb{R}^{r \times k}$, $B \in \mathbb{R}^{d \times r}$, $r \ll \min(d,k)$ — ранг. Параметров: $r(d+k)$ вместо $dk$.

**Плюсы:** мало обучаемых параметров, можно подключать разные LoRA-адаптеры к одной базовой модели, быстрая смена задач.

### Параметры LoraConfig (подробно)

#### **r** (rank) — ранг разложения

В формуле $W' = W + BA$ матрица $A$ имеет размер $r \times k$, матрица $B$ — $d \times r$. Ранг $r$ задаёт «толщину» низкорангового обновления: сколько скрытых размерностей мы добавляем к исходному слою.

- **Меньше r** (4, 8): меньше параметров, быстрее обучение, ниже риск переобучения, но меньше выразительности. Подходит для простых задач или очень малых датасетов.
- **Больше r** (32, 64): адаптер гибче, но больше параметров и выше риск переобучения. Уместен для сложных задач и достаточного объёма данных.
- **Типичные значения**: 8–32. В оригинальной статье LoRA часто используют r=8 для многих задач.

Количество обучаемых параметров LoRA в одном линейном слое: $r \cdot d + r \cdot k = r(d+k)$. При $d=k=4096$ и $r=16$ это $131\,072$ вместо $16\,777\,216$ полного обновления (примерно 0.8%).

---

#### **lora_alpha** — коэффициент масштабирования

В LoRA итоговое обновление умножается на коэффициент $\alpha/r$. В коде реализовано как:
$$h_{out} = h_{in} \cdot W + \frac{\alpha}{r} \cdot (h_{in} \cdot B \cdot A)$$

- **Зачем**: при маленьком $r$ обновление $BA$ может быть слишком слабым относительно исходного $W$. Множитель $\alpha/r$ позволяет усилить влияние LoRA.
- **Типичный выбор**: $\alpha = 2r$ (например, r=16 → alpha=32). Часто в статьях используют именно такое соотношение.
- **Большой alpha**: LoRA сильнее влияет на выход, обучение может быть нестабильнее.
- **Маленький alpha**: адаптер почти не меняет модель, обучение может быть слишком медленным.

---

#### **lora_dropout** — «микроинсульт» в LoRA-слоях

Можно представить так: во время обучения часть нейронов в LoRA-адаптере случайно «отключается» на каждом шаге — как микроинсульт: связь временно не передаётся. Параметр задаёт долю таких отключений (0.05 = 5%, 0.1 = 10%).

**Зачем:** если модель привыкает к одному пути и «запоминает» примеры, при отключении части связей ей приходится опираться на остальные. Это мешает заучиванию и помогает лучше обобщать.

- **0**: «микроинсульты» отключены, модель может сильнее переобучаться.
- **0.05–0.1**: умеренная доля отключений, обычно хороший диапазон.
- **Сильно больше 0.1**: слишком много «выключенных» связей, обучение может ухудшиться.

---

#### **bias** — обучать ли смещение (bias)

В линейном слое $y = Wx + b$ есть вектор $b$. Параметр задаёт, обновлять ли bias при дообучении:

- **'none'**: bias не обучается, обычно достаточно.
- **'lora_only'**: обучается только bias в LoRA-слоях.
- **'all'**: обучаются все bias в целевых слоях. Редко нужно, увеличивает число параметров.

---

#### **task_type** — тип задачи

Указывает, к какому виду языковой модели применяется LoRA:

- **CAUSAL_LM** (causal language model): модель предсказывает следующий токен по предыдущим (GPT, LLaMA, Qwen). Подходит для генерации текста, чат-моделей, дообучения на диалогах.
- **SEQ_2_SEQ_LM**: модель encoder–decoder (T5, BART), задачи типа «вход → выход».

---

#### **target_modules** — куда «подвешивать» LoRA

Список имён модулей, к чьим **линейным слоям** добавляется LoRA. Имена зависят от конкретной архитектуры.

**Transformer-блок (LLaMA, Qwen, GPT):**
- **q_proj**, **k_proj**, **v_proj** — проекции Query, Key, Value в self-attention. Преобразуют вход в представления для механизма внимания.
- **o_proj** — выходная проекция attention (объединяет головы).
- **gate_proj**, **up_proj**, **down_proj** — три матрицы в MLP (полносвязный блок после attention): gate и up образуют промежуточную активацию, down — финальный выход.

**Типичные комбинации:**
- Только attention: `['q_proj','k_proj','v_proj','o_proj']` — часто хватает для текстовых задач, меньше параметров.
- Attention + MLP: добавляют `gate_proj`, `up_proj`, `down_proj` — больше выразительности, больше параметров.
- Имена модулей можно посмотреть в `model.named_modules()`.

### BLEU: что это и как работает

**BLEU (Bilingual Evaluation Understudy)** — метрика качества машинного перевода и текстовой генерации, предложенная в Papineni et al. (2002). Сравнивает гипотезу (сгенерированный текст) с эталоном (референс).

**Идея:** считают n-граммное совпадение. Для n = 1, 2, 3, 4 вычисляют precision для каждой длины: сколько n-грамм из гипотезы встречается в референсе. Итоговый BLEU — геометрическое среднее этих precision с кратким штрафом за слишком короткие гипотезы.

**Формула (упрощённо):**
$$BLEU = BP \cdot \exp\Big(\sum_{n=1}^{4} w_n \ln p_n\Big)$$
где $p_n$ — precision для n-грамм, $w_n = 1/4$, BP (brevity penalty) — штраф, если гипотеза короче референса.

**Примеры:**

| Референс | Гипотеза | BLEU (примерно) | Почему |
|----------|----------|-----------------|--------|
| *Behold yon attire! Thou dost appear most splendid.* | *Behold yon attire! Thou dost appear most splendid.* | 1.0 | Полное совпадение |
| *Behold yon attire! Thou dost appear most splendid.* | *Behold that dress! You look splendid.* | ~0.5 | Часть слов совпадает |
| *Behold yon attire! Thou dost appear most splendid.* | *Look at that drip! You look fab.* | ~0.2 | Другая лексика |
| *Behold yon attire!* | *Behold yon attire!* | 1.0* | brevity penalty при разных длинах |

Синонимы не считаются совпадением — BLEU смотрит только на точные токены.

## Часть 1: Установка и baseline (2 балла)

**1 балл** — самостоятельно найти и выбрать модель, подходящую для квантизации (4-bit). Модель должна поддерживать чат-интерфейс и генерацию текста. До 4B.

**1 балл** — получить baseline BLEU > 0 через промпт (с zero-shot или few-shot; оценивается функцией `evaluate_model` ниже).

In [ ]:
!pip install -q -U torch transformers peft datasets bitsandbytes trl accelerate evaluate

import torch, gc, warnings
warnings.filterwarnings('ignore')
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, GenerationConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import evaluate

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
def cleanup(): gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.1 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4


---
## Часть 2: Подготовка (дано)

Данные, модель и базовая оценка приведены ниже — выполните эти ячейки перед частями 3 и 4.

In [ ]:
# 1. Датасет
dataset = load_dataset('harpreetsahota/modern-to-shakesperean-translation')
dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)

# 2. Токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# 3. Общий формат диалога (modern → shakespearean)
def make_messages(modern, shakespearean=None):
    msgs = [
        {"role": "system", "content": "You rewrite modern text in Shakespearean style."},
        {"role": "user", "content": f"Rewrite: {modern}"},
    ]
    if shakespearean is not None:
        msgs.append({"role": "assistant", "content": shakespearean})
    return msgs

def format_example(ex):
    return {'text': tokenizer.apply_chat_template(make_messages(ex['modern'], ex['shakespearean']), tokenize=False)}

train_dataset = dataset['train'].map(format_example)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/274 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/219 [00:00<?, ? examples/s]

In [ ]:
# 4. Загрузка модели и оценка
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
bleu_metric = evaluate.load('bleu')

def evaluate_model(model, tokenizer, test_data, n=15):
    model.eval()
    dev = next(model.parameters()).device
    subset = test_data.select(range(min(n, len(test_data))))
    gen_config = GenerationConfig(max_new_tokens=80, do_sample=True, temperature=0.6, pad_token_id=tokenizer.pad_token_id)
    preds = []
    for x in subset:
        prompt = tokenizer.apply_chat_template(make_messages(x['modern']), tokenize=False, add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors='pt').to(dev)
        with torch.no_grad():
            out = model.generate(**inp, generation_config=gen_config, max_new_tokens=80)
        gen = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        preds.append(gen)
    refs = [[x['shakespearean']] for x in subset]
    return bleu_metric.compute(predictions=preds, references=refs)['bleu'], preds, refs

baseline_bleu, baseline_preds, _ = evaluate_model(model, tokenizer, dataset['test'])
print(f'Baseline BLEU: {baseline_bleu:.4f}')

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Baseline BLEU: 0.0402


In [ ]:
# Автопроверка Часть 1 (2 балла): 1 — выбор модели, 2 — baseline_bleu > 0
_part1_pts = 0
try:
    model_ok = MODEL_ID and MODEL_ID.strip() and 'ВАША МОДЕЛЬ' not in MODEL_ID
    bleu_ok = 'baseline_bleu' in dir() and baseline_bleu > 0
    if model_ok:
        _part1_pts = 1
    if model_ok and bleu_ok:
        _part1_pts = 2
except Exception:
    pass
print(f"Часть 1: {_part1_pts}/2 баллов")

Часть 1: 2/2 баллов


---
## Часть 3: Тонкая настройка LoRA (3 балла)

**Критерии:** 1 балл — обучение запустилось (метрика может быть 0); 2 балла — метрика > 0; 3 балла — метрика > 0.15.

Промпт ипользовать только этот:
{"role": "system", "content": "You rewrite modern text in Shakespearean style."},
{"role": "user", "content": f"Rewrite: {modern}"},

**Задача.** Настроить LoRA и обучить модель.

1. Освободите память и загрузите свежую модель (ячейка ниже).
2. Создайте `LoraConfig` и примените LoRA через `get_peft_model`.
3. Настройте `SFTTrainer` и обучите модель.
4. Оцените модель через `evaluate_model` и сохраните результат в `lora_bleu`.

**Параметры LoraConfig** — см. пояснительную записку выше. Вопрос выбора значений остаётся открытым.

In [ ]:
cleanup(); del model
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# 1. LoraConfig и get_peft_model
# 2. model.print_trainable_parameters()

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,202,496 || all params: 498,235,264 || trainable%: 0.8435


In [ ]:
# 3. Обучение: SFTConfig, SFTTrainer, trainer.train()

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./lora_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss
10,2.736222
20,1.654375
30,1.480029
40,1.283828
50,1.158920
60,1.132733
70,0.999129
80,0.962433


TrainOutput(global_step=84, training_loss=1.402975340684255, metrics={'train_runtime': 94.6138, 'train_samples_per_second': 6.944, 'train_steps_per_second': 0.888, 'total_flos': 91114651789824.0, 'train_loss': 1.402975340684255})

In [ ]:
# 4. Оценка: evaluate_model → lora_bleu

lora_bleu, _, _ = evaluate_model(model, tokenizer, dataset['test'])
print(f'LoRA BLEU: {lora_bleu:.4f}')

LoRA BLEU: 0.1582


In [ ]:
# Автопроверка LoRA (3 балла): 1 — обучение запустилось, 2 — BLEU>0, 3 — BLEU>0.15
_lora_pts = 0
try:
    from peft import PeftModel
    if 'model' in dir() and isinstance(model, PeftModel) and 'lora_bleu' in dir():
        _lora_pts = 1
        if lora_bleu > 0:
            _lora_pts = 2
        if lora_bleu > 0.15:
            _lora_pts = 3
except Exception:
    pass
print(f"LoRA: {_lora_pts}/3 баллов")

LoRA: 3/3 баллов


---
## Часть 4: Тонкая настройка QLoRA (3 балла)

**Критерии:** 2 балла — обучение запустилось и метрика > 0; 3 балла — метрика > 0.07.

Промпт ипользовать только этот:
{"role": "system", "content": "You rewrite modern text in Shakespearean style."},
{"role": "user", "content": f"Rewrite: {modern}"},

**Задача.** QLoRA = квантизация модели в 4-bit + LoRA. Реализуйте:
1. Создайте `BitsAndBytesConfig` и загрузите модель с квантизацией.
2. Вызовите `prepare_model_for_kbit_training`, затем `get_peft_model` с тем же `lora_config`.
3. Обучите через SFTTrainer (важно: оптимизатор для квантизованных моделей).
4. Оцените и сохраните в `qlora_bleu`.

In [ ]:
cleanup()
try: del model
except NameError: pass
try: del trainer
except NameError: pass

In [ ]:
# BitsAndBytesConfig → загрузка модели → prepare_model_for_kbit_training → get_peft_model(lora_config)

from peft import prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 4,202,496 || all params: 498,235,264 || trainable%: 0.8435


In [ ]:
# Обучение QLoRA: SFTConfig и SFTTrainer

training_args = SFTConfig(
    output_dir="./qlora_out",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.717076
20,1.647645
30,1.477965
40,1.273912
50,1.150657
60,1.120442
70,0.982636
80,0.948936


TrainOutput(global_step=84, training_loss=1.3918341398239136, metrics={'train_runtime': 109.2355, 'train_samples_per_second': 6.015, 'train_steps_per_second': 0.769, 'total_flos': 91114651789824.0, 'train_loss': 1.3918341398239136})

In [ ]:
# Оценка: evaluate_model → qlora_bleu

qlora_bleu, _, _ = evaluate_model(model, tokenizer, dataset['test'])
print(f'QLoRA BLEU: {qlora_bleu:.4f}')

QLoRA BLEU: 0.1274


---
## Итоговое сравнение

In [ ]:
# Автопроверка QLoRA (3 балла): 2 — BLEU>0, 3 — BLEU>0.07
_qlora_pts = 0
try:
    _ns = get_ipython().user_ns if 'get_ipython' in dir() else globals()
except: _ns = globals()
try:
    qb = _ns.get('qlora_bleu')
    if qb is not None:
        if qb > 0:
            _qlora_pts = 2
        if qb > 0.07:
            _qlora_pts = 3
except Exception:
    pass
print(f"QLoRA: {_qlora_pts}/3 баллов")

QLoRA: 3/3 баллов


In [ ]:
print('='*60)
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ')
print('='*60)
results = {'Baseline': baseline_bleu, 'LoRA': lora_bleu, 'QLoRA': qlora_bleu}
print(f'\n{"Метод":<15} {"BLEU":<10} {"vs Baseline":<15}')
print('-'*40)
for method, bleu in results.items():
    imp = f'+{((bleu-baseline_bleu)/baseline_bleu*100):.0f}%' if baseline_bleu > 0 and bleu > baseline_bleu else '-'
    print(f'{method:<15} {bleu:.4f}     {imp}')
print(f'\nЛучший: {max(results, key=results.get)}')

ИТОГОВЫЕ РЕЗУЛЬТАТЫ

Метод           BLEU       vs Baseline    
----------------------------------------
Baseline        0.0402     -
LoRA            0.1582     +293%
QLoRA           0.1274     +217%

Лучший: LoRA


---
## Вывод (2 балла).

Вывод будет учитываться если в задании 3-4 были получены метрики отличные от 0. Иначе вывод не проверяется.

Напишите выводы **только своими словами, без использования GPT и нейросетей.** Опишите проведённые эксперименты и обоснуйте выбор параметров (rank, lora_alpha, target_modules и т.п.). Качественный вывод позволяет поднять итоговую оценку с 8 до 9–10 баллов.

⚠️ **Если в этом пункте будет обнаружен текст, сгенерированный GPT или аналогичными системами — за всю лабораторную выставляется оценка 0.** Проверка на GPT касается исключительно этого пункта; для частей LoRA и QLoRA она не применяется.

In [ ]:
# Текущий набранный балл (авто)
_auto = (_part1_pts if '_part1_pts' in dir() else 0) + (_lora_pts if '_lora_pts' in dir() else 0) + (_qlora_pts if '_qlora_pts' in dir() else 0)
print(f"\n{'='*40}")
print(f"ТЕКУЩИЙ БАЛЛ: {_auto}/8 (Часть1+LoRA+QLoRA) + до 2 вывод = макс 10")

*Напишите выводы ниже:* Lora показала себя лучше, чем QLora. Вообще, начнем с того, что относительно начального решения, я изменила сам предположительный ранг матрицы с 16 на 8 + увеличила количество эпох и обе модели выдали результат лучше. Проблема второй скорее в квантовании, ибо, очевидно, такое снижение размера влияет на точность весов и добавляет шум. На больших моделях это не так критично, но я взяла довольно маленькую.